# NB1 - Canonical feature ablation + parameter-matched MLP + linear baseline (Kaggle T4)

**Settings:** Accelerator = GPU T4, Internet = ON.  **Run:** Run All (or Save Version > Save & Run All).

**Datasets to attach (Add Data):**
1. `KAN_ROUTE_PART4_JOURNAL_READY.zip` (upload as a dataset)  - splits, human benchmark, canonical checkpoints and predictions
2. `kan-qwen`  (contains clip_features.npz)
3. `llama-kan` (contains clip_features.npz)
4. `images-new-known` (folders `known/` and `new/`)

**What it does:** freezes + checksums the canonical evidence; rebuilds the 896-d features and *proves* they reproduce the canonical predictions (parity gate); trains 18 Text-only models, re-evaluates the 18 canonical Text+Pixel models, trains 9 parameter-matched MLPs and 9 linear heads (3 seeds each); writes tables, figures and computed verdicts to `/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS/` and `/kaggle/working/NB1_OUTPUT.zip`.
Training is resumable: if the session dies, run again and finished checkpoints are re-used.

In [ ]:
# ===== NB1 | Setup =====  (needs Internet ON for pip + GitHub + Hugging Face weights, and GPU = T4)
import os, sys, re, json, time, math, shutil, random, subprocess, warnings, gc, hashlib
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

def pip_install(*pk):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk], check=False)
pip_install("open_clip_torch", "sentence-transformers")

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
assert torch.cuda.is_available(), "Turn on a GPU: Settings > Accelerator > GPU T4."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| torch", torch.__version__, "| cuda", torch.version.cuda)
if "T4" not in GPU_NAME:
    print("WARNING: this is not a T4. Results are still valid; only latency numbers (NB3) must be re-run on a T4.")

FASTKAN_DIR = Path("/kaggle/working/fast-kan")
if not FASTKAN_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ZiyaoLi/fast-kan.git", str(FASTKAN_DIR)], check=True)
sys.path.append(str(FASTKAN_DIR))
from fastkan import FastKAN

def find_jr_base():
    for p in Path("/kaggle/input").rglob("PART4_FINAL_REPRODUCIBILITY_AUDIT.md"):
        d = p.parent
        if (d / "part4_predictions_seed_corrected").is_dir() and (d / "qwen_shared28_train.csv").exists():
            return d
    raise FileNotFoundError("Attach KAN_ROUTE_PART4_JOURNAL_READY.zip as a Kaggle dataset (Add Data > Upload).")
JR = find_jr_base()
print("Canonical package:", JR)

OUT = Path("/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS")
D00 = OUT / "00_frozen_baseline"
D01 = OUT / "01_canonical_feature_ablation"
D02 = OUT / "02_parameter_matched_mlp"
D03 = OUT / "03_linear_baseline"
for d in (D00, D01, D02, D03):
    d.mkdir(parents=True, exist_ok=True)
    if d != D00:
        (d / "predictions").mkdir(exist_ok=True); (d / "checkpoints").mkdir(exist_ok=True)

# ---- run options ----
RETRAIN_TEXT_PIXEL = False   # False: reuse the 18 canonical Text+Pixel checkpoints (re-evaluated, same protocol/seeds). True: retrain all 18 (literal "36 new models").
DETERMINISM_CHECK = True     # retrain ONE canonical Text+Pixel config and report prediction agreement with the canonical checkpoint
MLP_PM_PROJ = 61             # projection width of the parameter-matched MLP (hidden widths stay 64,32) -> 62,003 params (+0.56% vs KAN 61,660)
SEEDS = [42, 123, 2026]


In [ ]:
# ======================= LIB_EVAL (pure numpy/pandas/sklearn; no torch) =======================
import json, re, math, hashlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

SEEDS = [42, 123, 2026]
N_CLASSES = 28
ALL_LABELS = list(range(N_CLASSES))

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def normalize_q(q):
    return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

def macro_f1_fast(y, p, labels):
    """Macro-F1 over `labels` (zero-division -> 0). Equivalent to sklearn f1_score(average='macro', labels=labels)."""
    labels = np.asarray(labels)
    k = int(max(y.max(), p.max(), labels.max())) + 1
    tp = np.bincount(y[y == p], minlength=k).astype(float)
    pc = np.bincount(p, minlength=k).astype(float)
    yc = np.bincount(y, minlength=k).astype(float)
    denom = pc + yc
    f1 = np.divide(2 * tp, denom, out=np.zeros(k), where=denom > 0)
    return float(f1[labels].mean())

# ---------------------------------- prediction files ----------------------------------
def read_pred(path):
    df = pd.read_csv(path)
    df["_top3"] = [json.loads(s) for s in df["pred_top3"]]
    return df

def top3_hit(y, top3_list):
    return np.fromiter((g in t for g, t in zip(y, top3_list)), bool, len(y))

def metrics_from_arrays(y, p, hit3, labels):
    if len(y) == 0:
        return dict(n=0, top1=np.nan, top3=np.nan, macro_f1=np.nan, weighted_f1=np.nan)
    return dict(n=int(len(y)), top1=float((y == p).mean()), top3=float(hit3.mean()),
                macro_f1=macro_f1_fast(y, p, labels),
                weighted_f1=float(f1_score(y, p, average="weighted", labels=labels, zero_division=0)))

# ---------------------------------- human benchmark ----------------------------------
class HumanBench:
    """Loads human_shared28.csv and derives all strata. Fixes the NaN->'nan' bug (acceptable set empty for strict rows)."""
    def __init__(self, human_csv, qtr_csv, ltr_csv, mapping_json):
        h = pd.read_csv(human_csv).reset_index(drop=True)
        self.map = json.load(open(mapping_json))["class_to_local_id"]
        self.inv = {v: k for k, v in self.map.items()}; self.inv[N_CLASSES] = "__INVALID__"
        h["acceptable_tool_labels"] = h["acceptable_tool_labels"].fillna("").astype(str).str.strip()
        strict_true = h["strict_eval"].astype(str).str.lower().isin(["true", "1"]).values
        self.is_amb = strict_true & (h["acceptable_tool_labels"] != "").values
        self.is_strict = strict_true & ~self.is_amb
        assert self.is_strict.sum() + self.is_amb.sum() == len(h)
        self.h = h
        qtr, ltr = pd.read_csv(qtr_csv), pd.read_csv(ltr_csv)
        train_imgs = set(qtr.img_id) | set(ltr.img_id)
        self.known = h["image_id"].isin(train_imgs).values
        assert (self.known == (h["image_pool"] == "known").values).all(), "known/new recomputation disagrees with image_pool"
        self.vis_yes = (h["routing_requires_visual_context"].astype(str).str.lower() == "yes").values
        self.y = h["tool_name"].map(self.map).values.astype(int)
        self.acc_sets = [set(x.strip() for x in s.split(",") if x.strip()) for s in h["acceptable_tool_labels"]]
        self.strata = {"all": np.ones(len(h), bool), "vis_yes": self.vis_yes, "vis_no": ~self.vis_yes,
                       "known": self.known, "new": ~self.known}
        shared = sorted(set(self.y[self.known]) & set(self.y[~self.known]))
        self.shared_labels = shared

    def eval_human(self, df, use_bench_labels=False):
        assert len(df) == len(self.h) and (df["sample_id"].astype(str).values == self.h["query_id"].astype(str).values).all(), \
            "human prediction rows not aligned with query_id"
        y, p, t3 = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int), df["_top3"].values
        if use_bench_labels:
            y = self.y.copy()
        else:
            assert (y == self.y).all()
        hit3 = top3_hit(y, t3)
        top1_names = [self.inv[i] for i in p]
        amb1 = np.array([top1_names[i] in self.acc_sets[i] for i in range(len(y))])
        amb3 = np.array([bool({self.inv[c] for c in t3[i]} & self.acc_sets[i]) for i in range(len(y))])
        rows = []
        for name, m in self.strata.items():
            labs = ALL_LABELS if name == "all" else sorted(set(y[m]))
            r = metrics_from_arrays(y[m], p[m], hit3[m], labs)
            r["macro_f1_label_set"] = "28" if name == "all" else "present"
            if name in ("known", "new"):
                r["macro_f1_shared26"] = macro_f1_fast(y[m], p[m], self.shared_labels)
            sm, am = m & self.is_strict, m & self.is_amb
            r.update(strict_n=int(sm.sum()), strict_top1=float((y[sm] == p[sm]).mean()) if sm.any() else np.nan,
                     strict_top3=float(hit3[sm].mean()) if sm.any() else np.nan,
                     ambig_n=int(am.sum()), ambig_top1=float(amb1[am].mean()) if am.any() else np.nan,
                     ambig_top3=float(amb3[am].mean()) if am.any() else np.nan)
            r["cell"] = f"human_{name}"
            rows.append(r)
        return rows

class SyntheticBench:
    """Image-disjoint masks for the two teacher test sets."""
    def __init__(self, qtr_csv, qte_csv, ltr_csv, lte_csv):
        self.qtr, self.qte, self.ltr, self.lte = [pd.read_csv(p) for p in (qtr_csv, qte_csv, ltr_csv, lte_csv)]
        qkeys, lkeys = set(zip(self.qtr.img_id, self.qtr.tool)), set(zip(self.ltr.img_id, self.ltr.tool))
        self.disj = {"qwen_test": np.array([(i, t) not in lkeys for i, t in zip(self.qte.img_id, self.qte.tool)]),
                     "llama_test": np.array([(i, t) not in qkeys for i, t in zip(self.lte.img_id, self.lte.tool)])}

    def eval_synth(self, df, which):
        y, p, t3 = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int), df["_top3"].values
        hit3 = top3_hit(y, t3)
        out = []
        for suffix, m in (("", np.ones(len(y), bool)), ("_disjoint", self.disj[which])):
            r = metrics_from_arrays(y[m], p[m], hit3[m], ALL_LABELS)
            r["cell"] = which + suffix
            out.append(r)
        return out

def eval_prediction_set(preds, sbench, hbench):
    """preds: dict with keys qwen_test, llama_test, human -> prediction DataFrames. Returns list of metric rows (cells)."""
    rows = []
    rows += sbench.eval_synth(preds["qwen_test"], "qwen_test")
    rows += sbench.eval_synth(preds["llama_test"], "llama_test")
    rows += hbench.eval_human(preds["human"])
    return rows

def per_class_f1(df):
    y, p = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int)
    k = N_CLASSES
    tp = np.bincount(y[y == p], minlength=k).astype(float); pc = np.bincount(p, minlength=k).astype(float); yc = np.bincount(y, minlength=k).astype(float)
    d = pc + yc
    return np.divide(2 * tp, d, out=np.zeros(k), where=d > 0), yc.astype(int)

# ---------------------------------- aggregation ----------------------------------
def sd(x):
    x = np.asarray(x, float)
    return float(np.std(x, ddof=1)) if len(x) > 1 else 0.0

def aggregate_seeds(rows_df, keys, metrics):
    out = []
    for k, g in rows_df.groupby(keys, sort=False):
        rec = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        for m in metrics:
            v = g.sort_values("seed")[m].values.astype(float)
            rec[f"{m}_mean"], rec[f"{m}_sd"] = float(np.nanmean(v)), sd(v)
            for s, val in zip(sorted(g["seed"].values), v):
                rec[f"{m}_seed{s}"] = float(val)
        rec["n_seeds"] = int(g["seed"].nunique())
        out.append(rec)
    return pd.DataFrame(out)

def paired_delta(rows_df, id_cols, factor, a, b, metrics):
    """Seed-paired delta = metric(factor==a) - metric(factor==b); returns per-seed rows."""
    A = rows_df[rows_df[factor] == a].set_index(id_cols + ["seed"])
    B = rows_df[rows_df[factor] == b].set_index(id_cols + ["seed"])
    idx = A.index.intersection(B.index)
    d = pd.DataFrame(index=idx)
    for m in metrics:
        d[m] = A.loc[idx, m].values - B.loc[idx, m].values
    return d.reset_index()

# ---------------------------------- statistics ----------------------------------
def mcnemar_exact(correct_a, correct_b):
    from scipy.stats import binomtest
    b = int((correct_a & ~correct_b).sum()); c = int((~correct_a & correct_b).sum())
    n = b + c
    p = 1.0 if n == 0 else float(binomtest(min(b, c), n, 0.5, alternative="two-sided").pvalue)
    return b, c, p

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def paired_bootstrap(preds_a, preds_b, y, B=1000, seed=0, labels=ALL_LABELS):
    """preds_*: list over seeds of int arrays on identical rows. Returns CIs of seed-averaged delta Top-1 and delta macro-F1."""
    rng = np.random.RandomState(seed); n = len(y)
    d1, df1 = [], []
    for _ in range(B):
        idx = rng.randint(0, n, n); yy = y[idx]
        t = np.mean([(pa[idx] == yy).mean() - (pb[idx] == yy).mean() for pa, pb in zip(preds_a, preds_b)])
        f = np.mean([macro_f1_fast(yy, pa[idx], labels) - macro_f1_fast(yy, pb[idx], labels) for pa, pb in zip(preds_a, preds_b)])
        d1.append(t); df1.append(f)
    q = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return q(d1), q(df1)

# ---------------------------------- KAN edge statistics ----------------------------------
def kan_edge_stats(spline_w, in_dim, G=4, lo=-2.0, hi=2.0, npts=200):
    """spline_w: (out_dim, in_dim*G) FastKAN spline_linear weight. Returns per-edge arrays."""
    out_dim = spline_w.shape[0]
    centers = np.linspace(lo, hi, G); rho = (hi - lo) / (G - 1)
    v = np.linspace(lo, hi, npts)
    Bm = np.exp(-(((v[:, None] - centers[None, :]) / rho) ** 2))               # (npts,G)
    W = spline_w.reshape(out_dim, in_dim, G)
    S = np.einsum("pg,oig->oip", Bm, W).reshape(-1, npts)                       # (edges,npts)
    amp = S.max(1) - S.min(1)
    vc = v - v.mean(); sxx = (vc ** 2).sum()
    slope = (S - S.mean(1, keepdims=True)) @ vc / sxx
    fit_ss = (slope ** 2) * sxx
    tot_ss = ((S - S.mean(1, keepdims=True)) ** 2).sum(1)
    r2 = np.divide(fit_ss, tot_ss, out=np.ones_like(fit_ss), where=tot_ss > 1e-12)
    dS = np.diff(S, axis=1)
    mono = (dS >= -1e-9).all(1) | (dS <= 1e-9).all(1)
    return dict(amp=amp.reshape(out_dim, in_dim), r2=r2, mono=mono, out_dim=out_dim, in_dim=in_dim)

def edge_summary(st, topks=(3, 10)):
    r = dict(edges=int(st["r2"].size), near_linear_share=float((st["r2"] >= 0.95).mean()), median_r2=float(np.median(st["r2"])),
             monotone_share=float(st["mono"].mean()), median_amp=float(np.median(st["amp"])))
    A = st["amp"]
    for k in topks:
        if A.shape[1] > k:
            srt = -np.sort(-A, axis=1)
            r[f"top{k}_amp_share"] = float((srt[:, :k].sum(1) / srt.sum(1)).mean())
    return r

# ---------------------------------- LaTeX helpers ----------------------------------
def pm(mu, s, scale=100.0, nd=2):
    return f"{mu*scale:.{nd}f}$\\pm${s*scale:.{nd}f}"

def latex_table(df, caption, label, colfmt=None):
    cols = list(df.columns); colfmt = colfmt or ("l" * len(cols))
    esc = lambda x: str(x).replace("_", "\\_").replace("%", "\\%")
    L = ["\\begin{table}[t]", "\\centering", "\\small", f"\\begin{{tabular}}{{{colfmt}}}", "\\toprule",
         " & ".join(esc(c) for c in cols) + " \\\\", "\\midrule"]
    for _, r in df.iterrows():
        L.append(" & ".join(v if "$\\pm$" in str(v) else esc(v) for v in r.values) + " \\\\")
    L += ["\\bottomrule", "\\end{tabular}", f"\\caption{{{caption}}}", f"\\label{{{label}}}", "\\end{table}"]
    return "\n".join(L)

# ---------------------------------- annotation agreement ----------------------------------
def jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not a and not b else len(a & b) / len(a | b)

def parse_set(s):
    return sorted({x.strip() for x in str(s).replace(";", ",").split(",") if x.strip() and x.strip().lower() != "nan"})

def agreement_report(a, b, tool_col="chosen_tool", amb_col="is_ambiguous", vis_col="routing_requires_visual_context", set_col="acceptable_tool_labels"):
    from sklearn.metrics import cohen_kappa_score
    m = a.merge(b, on="query_id", suffixes=("_1", "_2"))
    res = {"n_items": len(m)}
    res["tool_raw_agreement"] = float((m[f"{tool_col}_1"] == m[f"{tool_col}_2"]).mean())
    res["tool_cohen_kappa"] = float(cohen_kappa_score(m[f"{tool_col}_1"], m[f"{tool_col}_2"]))
    for nm, col in (("ambiguity", amb_col), ("visual_context", vis_col)):
        x, y = m[f"{col}_1"].astype(str).str.lower().str.strip(), m[f"{col}_2"].astype(str).str.lower().str.strip()
        res[f"{nm}_raw_agreement"] = float((x == y).mean())
        res[f"{nm}_cohen_kappa"] = float(cohen_kappa_score(x, y)) if x.nunique() > 1 or y.nunique() > 1 else float("nan")
    s1 = [parse_set(v) for v in m[f"{set_col}_1"]]; s2 = [parse_set(v) for v in m[f"{set_col}_2"]]
    jac = np.array([jaccard(u, v) for u, v in zip(s1, s2)])
    res["acceptable_set_exact_agreement"] = float(np.mean([u == v for u, v in zip(s1, s2)]))
    res["acceptable_set_mean_jaccard"] = float(jac.mean()); res["acceptable_set_median_jaccard"] = float(np.median(jac))
    return res, m


# ---------------------------------- stable-support (21 classes) ----------------------------------
def stable_metrics(y, p, hit3, stable_ids):
    """Only queries whose gold class is stable; F1 averaged over the stable classes; predictions into other classes are errors."""
    m = np.isin(y, stable_ids)
    if not m.any():
        return dict(n=0, top1=np.nan, top3=np.nan, macro_f1=np.nan)
    return dict(n=int(m.sum()), top1=float((y[m] == p[m]).mean()), top3=float(hit3[m].mean()),
                macro_f1=macro_f1_fast(y[m], p[m], np.asarray(stable_ids)))

def add_stable_cells(rows, preds, stable_ids):
    for key, cell in (("qwen_test", "qwen_test_stable21"), ("llama_test", "llama_test_stable21"), ("human", "human_all_stable21")):
        df = preds[key]; y, p = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int)
        r = stable_metrics(y, p, top3_hit(y, df["_top3"].values), stable_ids); r["cell"] = cell; rows.append(r)
    return rows

# ---------------------------------- pooled evaluation groups (as in manuscript Table 13) ----------------------------------
GROUPS = {
    "within_teacher": [("qwen", "qwen_test"), ("llama", "llama_test"), ("mixed", "qwen_test"), ("mixed", "llama_test")],
    "cross_teacher": [("qwen", "llama_test"), ("llama", "qwen_test")],
    "image_disjoint": [("qwen", "llama_test_disjoint"), ("mixed", "llama_test_disjoint"), ("llama", "qwen_test_disjoint"), ("mixed", "qwen_test_disjoint")],
    "human_written": [("qwen", "human_all"), ("llama", "human_all"), ("mixed", "human_all")],
    "human_new_image": [("qwen", "human_new"), ("llama", "human_new"), ("mixed", "human_new")],
}

def pooled_group(rows_df, model_col, model, group, metric, groups=None):
    groups = groups or GROUPS
    """Per-seed mean over the group's (regime, cell) pairs -> array over SEEDS."""
    out = []
    for s in SEEDS:
        v = []
        for reg, cell in groups[group]:
            r = rows_df[(rows_df[model_col] == model) & (rows_df["regime"] == reg) & (rows_df["cell"] == cell) & (rows_df["seed"] == s)]
            v.append(float(r[metric].iloc[0]))
        out.append(np.mean(v))
    return np.array(out)

def pooled_compare(rows_df, model_col, a, b, metrics=("top1", "top3", "macro_f1"), groups=None):
    groups = groups or GROUPS
    res = []
    for g in groups:
        for m in metrics:
            va, vb = pooled_group(rows_df, model_col, a, g, m, groups), pooled_group(rows_df, model_col, b, g, m, groups)
            d = va - vb
            res.append(dict(group=g, metric=m, a=a, b=b, a_mean=va.mean(), a_sd=sd(va), b_mean=vb.mean(), b_sd=sd(vb),
                            diff_mean=d.mean(), diff_sd=sd(d), seeds_a_ahead=int((d > 0).sum())))
    return pd.DataFrame(res)

# ---------------------------------- zero-shot LLM output parsing ----------------------------------
REFUSAL_PAT = re.compile(r"(i cannot|i can't|i can not|i'm sorry|i am sorry|unable to|none of the (above|tools)|no (suitable|appropriate) tool|cannot determine)", re.I)

def _norm_tok(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def parse_tool(text, tools):
    """Deterministic parser. Returns (tool_or_None, status) with status in valid_exact/valid_parsed/multiple/refusal/invalid."""
    t = _norm_tok(text)
    if t in tools:
        return t, "valid_exact"
    names = sorted(tools, key=len, reverse=True)
    rest, found = f"_{t}_", []
    for n in names:                                     # longest first so a short name inside a longer one is not double-counted
        if f"_{n}_" in rest:
            found.append(n); rest = rest.replace(f"_{n}_", "__")
    if len(found) == 1:
        return found[0], "valid_parsed"
    if len(found) > 1:
        return None, "multiple"
    return None, ("refusal" if REFUSAL_PAT.search(str(text)) else "invalid")

def parse_top3(text, tools):
    """Ordered list of up to 3 distinct valid tool names in order of appearance."""
    found = []
    for part in re.split(r"[,\n;]| and ", str(text)):
        tool, st = parse_tool(part, tools)
        if tool is not None and tool not in found:
            found.append(tool)
    if not found:                                       # fall back to order-of-appearance scan of the whole text
        t = f"_{_norm_tok(text)}_"
        pos = sorted((t.find(f"_{n}_"), n) for n in tools if f"_{n}_" in t)
        found = [n for _, n in pos]
    return found[:3]


GROUPS_STABLE = {
    "within_teacher": [(r, c + "_stable21") for r, c in GROUPS["within_teacher"]],
    "cross_teacher": [(r, c + "_stable21") for r, c in GROUPS["cross_teacher"]],
    "human_written": [(r, "human_all_stable21") for r in ("qwen", "llama", "mixed")],
}

def latex_table_resized(df, caption, label):
    t = latex_table(df, caption, label)
    return t.replace("\\begin{tabular}", "\\resizebox{\\textwidth}{!}{%\n\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}}", 1)


In [ ]:
# ===== NB1 | Section 1: freeze + checksum the canonical evidence (nothing here is modified) =====
FROZEN = []
for pat in ["qwen_shared28_*.csv", "llama_shared28_*.csv", "human_shared28*.csv", "shared_28_class_mapping.json", "shared28_taxonomy.txt",
            "stable_support_classes.txt", "part4_all_results_seed_corrected.csv", "part4_three_seed_summary.csv",
            "part4_per_class_metrics_seed_corrected.csv", "kan_*_shared28_best.pt", "mlp_*_shared28_best.pt",
            "PART4_FINAL_REPRODUCIBILITY_AUDIT.md", "PART4_SCIENTIFIC_VERDICT_FINAL.md"]:
    FROZEN += sorted(JR.glob(pat))
FROZEN += sorted((JR / "checkpoints_seed_corrected").glob("*.pt")) + sorted((JR / "part4_predictions_seed_corrected").glob("*.csv"))
FROZEN = sorted(set(FROZEN))

def category(p):
    n, rel = p.name, str(p.relative_to(JR))
    if rel.startswith("checkpoints_seed_corrected") or n.endswith(".pt"): return "checkpoint"
    if rel.startswith("part4_predictions"): return "prediction"
    if n.endswith("_train.csv") or n.endswith("_val.csv") or n.endswith("_test.csv"): return "split"
    if n.startswith("human_shared28"): return "human_benchmark"
    if "mapping" in n or "taxonomy" in n or "stable_support" in n: return "taxonomy"
    return "summary_or_report"

man = pd.DataFrame([dict(category=category(p), relative_path=str(p.relative_to(JR)), bytes=p.stat().st_size, sha256=sha256_file(p)) for p in FROZEN])
man.to_csv(D00 / "00_frozen_manifest.csv", index=False)
(D00 / "00_sha256_manifest.txt").write_text("\n".join(f"{r.sha256}  {r.relative_path}" for r in man.itertuples()) + "\n")
print(man.groupby("category").size().to_string()); print("files frozen:", len(man))

# ---- independent re-check of the protocol invariants ----
mp = json.load(open(JR / "shared_28_class_mapping.json"))["class_to_local_id"]
checks = {"n_classes_is_28": len(mp) == 28, "ids_contiguous_0_27": sorted(mp.values()) == list(range(28)), "matting_absent": "matting" not in mp}
qtr, qva, qte = [pd.read_csv(JR / f"qwen_shared28_{s}.csv") for s in ("train", "val", "test")]
ltr, lva, lte = [pd.read_csv(JR / f"llama_shared28_{s}.csv") for s in ("train", "val", "test")]
def overlaps(a, b, c):
    n = [set(d["query"].map(normalize_q)) for d in (a, b, c)]; it = [set(zip(d.img_id, d.tool)) for d in (a, b, c)]
    return (len(n[0] & n[1]) + len(n[0] & n[2]) + len(n[1] & n[2]), len(it[0] & it[1]) + len(it[0] & it[2]) + len(it[1] & it[2]))
qo, lo = overlaps(qtr, qva, qte), overlaps(ltr, lva, lte)
checks.update(qwen_normalized_query_overlap=qo[0], qwen_image_tool_overlap=qo[1], llama_normalized_query_overlap=lo[0], llama_image_tool_overlap=lo[1],
              split_sizes=dict(qwen=[len(qtr), len(qva), len(qte)], llama=[len(ltr), len(lva), len(lte)]))
checks["all_invariants_hold"] = bool(checks["n_classes_is_28"] and checks["ids_contiguous_0_27"] and checks["matting_absent"] and qo == (0, 0) and lo == (0, 0))
json.dump(checks, open(D00 / "00_frozen_checks.json", "w"), indent=2)
json.dump(dict(gpu=GPU_NAME, torch=torch.__version__, cuda=torch.version.cuda, python=sys.version, pandas=pd.__version__, numpy=np.__version__),
          open(D00 / "00_environment.json", "w"), indent=2)
print(json.dumps(checks, indent=2)); assert checks["all_invariants_hold"], "Canonical protocol invariants violated - stop."


In [ ]:
# ===== NB1 | Rebuild the canonical 896-d features (MiniLM + real CLIP pixels), exactly as in the canonical pipeline =====
from sentence_transformers import SentenceTransformer
import open_clip
from PIL import Image

minilm = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))
def load_clip_cache(tag):
    p = next(p for p in Path("/kaggle/input").rglob("clip_features.npz") if tag in str(p).lower())
    z = np.load(p, allow_pickle=True); return {int(k): v for k, v in z[z.files[0]].item().items()}
qwen_clip, llama_clip = load_clip_cache("qwen"), load_clip_cache("llama")
print("clip caches:", len(qwen_clip), len(llama_clip))

def text_embed(texts):
    return minilm.encode(list(texts), batch_size=256, convert_to_numpy=True, normalize_embeddings=False, show_progress_bar=False).astype(np.float32)
def feats(df, cache):
    return np.concatenate([text_embed(df["query"]), np.stack([cache[int(i)] for i in df["img_id"]]).astype(np.float32)], axis=1)

clip_model, _, clip_pre = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai", force_quick_gelu=True)
clip_model = clip_model.to(DEVICE).eval()
def find_img_dirs():
    k = [p for p in Path("/kaggle/input").rglob("known") if p.is_dir() and any(p.glob("*.jpg"))]
    n = [p for p in Path("/kaggle/input").rglob("new") if p.is_dir() and any(p.glob("*.jpg"))]
    if not k or not n: raise FileNotFoundError("Attach the images dataset that contains known/ and new/ folders.")
    return k[0], n[0]
KNOWN_DIR, NEW_DIR = find_img_dirs()
def img_path(i):
    for d in (KNOWN_DIR, NEW_DIR):
        p = d / f"{i}.jpg"
        if p.exists(): return p
    raise FileNotFoundError(f"image {i}")
@torch.inference_mode()
def clip_image_vec(path):
    x = clip_pre(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)
    f = clip_model.encode_image(x); f = f / f.norm(dim=-1, keepdim=True)
    return f.float().cpu().numpy()[0]

hb = HumanBench(JR / "human_shared28.csv", JR / "qwen_shared28_train.csv", JR / "llama_shared28_train.csv", JR / "shared_28_class_mapping.json")
sb = SyntheticBench(JR / "qwen_shared28_train.csv", JR / "qwen_shared28_test.csv", JR / "llama_shared28_train.csv", JR / "llama_shared28_test.csv")
TOOL2ID = hb.map; ID2TOOL = {v: k for k, v in TOOL2ID.items()}
STABLE = [TOOL2ID[c] for c in (JR / "stable_support_classes.txt").read_text().split()]
print("stable-support classes:", len(STABLE))

splits = {"qwen": dict(train=qtr, val=qva, test=qte, cache=qwen_clip), "llama": dict(train=ltr, val=lva, test=lte, cache=llama_clip)}
XS, YS = {}, {}
for t, d in splits.items():
    XS[t], YS[t] = {}, {}
    for s in ("train", "val", "test"):
        XS[t][s] = feats(d[s], d["cache"]); YS[t][s] = d[s]["tool"].map(TOOL2ID).values.astype(np.int64)
        print(t, s, XS[t][s].shape)
XH = np.concatenate([text_embed(hb.h["query"]), np.stack([clip_image_vec(img_path(i)) for i in hb.h["image_id"]]).astype(np.float32)], axis=1)
print("human", XH.shape)

REG_DATA = {
    "qwen": (XS["qwen"]["train"], YS["qwen"]["train"], XS["qwen"]["val"], YS["qwen"]["val"]),
    "llama": (XS["llama"]["train"], YS["llama"]["train"], XS["llama"]["val"], YS["llama"]["val"]),
    "mixed": (np.concatenate([XS["qwen"]["train"], XS["llama"]["train"]]), np.concatenate([YS["qwen"]["train"], YS["llama"]["train"]]),
              np.concatenate([XS["qwen"]["val"], XS["llama"]["val"]]), np.concatenate([YS["qwen"]["val"], YS["llama"]["val"]])),
}
print({k: (len(v[1]), len(v[3])) for k, v in REG_DATA.items()}, "(train, val rows)  expected qwen 30993/3875, llama 7101/936, mixed 38094/4811")
EVAL_X = {"qwen_test": XS["qwen"]["test"], "llama_test": XS["llama"]["test"], "human": XH}
EVAL_META = {
    "qwen_test": dict(id=np.arange(len(qte)), img=qte.img_id.values, query=qte["query"].values, tool=qte.tool.values, y=YS["qwen"]["test"]),
    "llama_test": dict(id=np.arange(len(lte)), img=lte.img_id.values, query=lte["query"].values, tool=lte.tool.values, y=YS["llama"]["test"]),
    "human": dict(id=hb.h.query_id.values, img=hb.h.image_id.values, query=hb.h["query"].values, tool=hb.h.tool_name.values, y=hb.y),
}


In [ ]:
# ===== NB1 | Models, seeding, training, prediction (identical protocol to the canonical run) =====
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

class KANRoute(nn.Module):
    def __init__(self, input_dim=896, proj_dim=48, hidden_dims=(48,), num_tools=28, num_grids=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim); self.norm = nn.LayerNorm(proj_dim)
        self.kan = FastKAN(layers_hidden=[proj_dim] + list(hidden_dims) + [num_tools], num_grids=num_grids)
    def forward(self, x): return self.kan(self.norm(self.proj(x.float())))

class MLPBaseline(nn.Module):
    def __init__(self, input_dim=896, proj_dim=96, hidden_dims=(64, 32), num_tools=28, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout)]
        prev = proj_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]; prev = h
        layers.append(nn.Linear(prev, num_tools))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x.float())

class LinearHead(nn.Module):
    def __init__(self, input_dim=896, num_tools=28):
        super().__init__(); self.fc = nn.Linear(input_dim, num_tools)
    def forward(self, x): return self.fc(x.float())

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

HP = {"kan": dict(lr=5e-4, epochs=80), "mlp": dict(lr=1e-3, epochs=60), "mlp_pm": dict(lr=1e-3, epochs=60), "linear": dict(lr=1e-3, epochs=60)}
def build_model(arch, input_dim=896):
    if arch == "kan": return KANRoute(input_dim=input_dim)
    if arch == "mlp": return MLPBaseline(input_dim=input_dim)
    if arch == "mlp_pm": return MLPBaseline(input_dim=input_dim, proj_dim=MLP_PM_PROJ)   # same widths (64,32) and dropout 0.3; only the projection is narrower
    if arch == "linear": return LinearHead(input_dim=input_dim)
    raise ValueError(arch)

class ToolRoutingDataset(Dataset):
    def __init__(self, X, y): self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]
def make_loader(X, y, shuffle): return DataLoader(ToolRoutingDataset(X, y), batch_size=256, shuffle=shuffle)

def train_model(model, tr, va, arch, ckpt_path, wd=1e-4):
    lr, epochs = HP[arch]["lr"], HP[arch]["epochs"]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(tr), epochs=epochs, pct_start=0.1)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best, best_ep = 0.0, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.float().to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        model.eval(); c = t = 0
        with torch.no_grad():
            for xb, yb in va:
                xb, yb = xb.float().to(DEVICE), yb.to(DEVICE)
                c += (model(xb).argmax(-1) == yb).sum().item(); t += len(yb)
        acc = c / t
        if acc > best: best, best_ep = acc, ep; torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return best, best_ep, epochs, lr

@torch.no_grad()
def predict_probs(model, X, bs=1024):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(torch.softmax(model(torch.from_numpy(X[i:i + bs]).float().to(DEVICE)), -1).cpu().numpy())
    return np.concatenate(out)

def make_pred_df(probs, ev, tags):
    m = EVAL_META[ev]; pred = probs.argmax(1); top3 = np.argsort(-probs, axis=1)[:, :3]
    df = pd.DataFrame({"sample_id": m["id"], "img_id": m["img"], "query": m["query"], "ground_truth_tool": m["tool"], "ground_truth": m["y"],
                       "pred_top1": pred, "pred_top1_tool": [ID2TOOL[int(i)] for i in pred],
                       "pred_top3": [json.dumps([int(v) for v in t]) for t in top3],
                       "probs": [";".join(f"{p:.6f}" for p in r) for r in probs], "correct": pred == m["y"], **tags})
    df["_top3"] = [[int(v) for v in t] for t in top3]
    return df

# parameter accounting (also validates that the parameter-matched MLP really is parameter-matched)
PARAMS = {a: n_params(build_model(a)) for a in ("kan", "mlp", "mlp_pm", "linear")}
PARAMS["kan_text_only"] = n_params(build_model("kan", 384)); PARAMS["mlp_text_only"] = n_params(build_model("mlp", 384))
print("trainable parameters:", PARAMS)
assert PARAMS["kan"] == 61660 and PARAMS["mlp"] == 95708, "canonical parameter counts changed"
assert abs(PARAMS["mlp_pm"] - PARAMS["kan"]) / PARAMS["kan"] <= 0.02, "MLP-PM is not within 2% of KAN"


In [ ]:
# ===== NB1 | Parity gate: canonical checkpoints on the REBUILT features must reproduce the saved canonical predictions =====
PARITY = []
for arch, regime, seed in [("kan", "mixed", 42), ("mlp", "qwen", 123), ("kan", "llama", 2026), ("mlp", "mixed", 2026)]:
    ck = JR / "checkpoints_seed_corrected" / f"{arch}_{regime}_shared28_seed{seed}_corrected_best.pt"
    m = build_model(arch, 896).to(DEVICE); m.load_state_dict(torch.load(ck, map_location=DEVICE))
    for ev in ("qwen_test", "llama_test", "human"):
        probs = predict_probs(m, EVAL_X[ev])
        saved = pd.read_csv(JR / "part4_predictions_seed_corrected" / f"{arch}_{regime}_seed{seed}_on_{ev}.csv")
        sp = np.array([[float(v) for v in s.split(";")] for s in saved["probs"]])
        PARITY.append(dict(model=f"{arch}_{regime}_seed{seed}", eval=ev, n=len(saved), label_agreement=float((probs.argmax(1) == saved["pred_top1"].values).mean()),
                           max_abs_prob_diff=float(np.abs(probs - sp).max())))
parity = pd.DataFrame(PARITY); parity.to_csv(D00 / "00_feature_parity_check.csv", index=False)
print(parity.to_string(index=False))
assert parity.label_agreement.min() >= 0.995, "Rebuilt features do not reproduce canonical predictions - fix the data path before training anything."


In [ ]:
# ===== NB1 | Train / evaluate every configuration (resumable: existing checkpoints are re-used) =====
FEAT_DIM = {"text_only": 384, "text_pixel": 896}
def feat_view(X, feat): return X if feat == "text_pixel" else X[:, :384]
LOADERS = {}
def get_loaders(feat, regime):
    k = (feat, regime)
    if k not in LOADERS:
        Xtr, ytr, Xva, yva = REG_DATA[regime]
        LOADERS[k] = (make_loader(feat_view(Xtr, feat), ytr, True), make_loader(feat_view(Xva, feat), yva, False))
    return LOADERS[k]

OUTDIR = {"kan": D01, "mlp": D01, "mlp_pm": D02, "linear": D03}
RES_ROWS, TRAIN_LOG, PC_ROWS = [], [], []

def run_config(feat, arch, regime, seed, from_ckpt=None, tag=""):
    name = f"{feat}_{arch}_{regime}_seed{seed}{tag}"
    d = OUTDIR[arch]; ckpt = d / "checkpoints" / f"{name}_best.pt"
    set_seed(seed)                                   # <-- seed BEFORE the model is constructed (weights depend on the seed)
    model = build_model(arch, FEAT_DIM[feat]).to(DEVICE)
    trained, best, best_ep, epochs, lr = False, np.nan, -1, HP[arch]["epochs"], HP[arch]["lr"]
    if from_ckpt is not None:
        model.load_state_dict(torch.load(from_ckpt, map_location=DEVICE)); shutil.copy(from_ckpt, ckpt)
    elif ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    else:
        tr, va = get_loaders(feat, regime)
        best, best_ep, epochs, lr = train_model(model, tr, va, arch, ckpt); trained = True
    preds = {}
    for ev in ("qwen_test", "llama_test", "human"):
        probs = predict_probs(model, feat_view(EVAL_X[ev], feat))
        df = make_pred_df(probs, ev, dict(feature_set=feat, architecture=arch, training_source=regime, evaluation_source=ev, seed=seed))
        df.drop(columns=["_top3"]).to_csv(d / "predictions" / f"{name}_on_{ev}.csv", index=False)
        preds[ev] = df
        f1, sup = per_class_f1(df)
        for c in range(28): PC_ROWS.append(dict(feat=feat, arch=arch, regime=regime, seed=seed, tag=tag, eval=ev, class_id=c, class_name=ID2TOOL[c], f1=float(f1[c]), support=int(sup[c])))
    rows = add_stable_cells(eval_prediction_set(preds, sb, hb), preds, STABLE)
    for r in rows: r.update(feat=feat, arch=arch, regime=regime, seed=seed, tag=tag)
    TRAIN_LOG.append(dict(feat=feat, arch=arch, regime=regime, seed=seed, params=n_params(model), trained_here=trained, best_val_acc=best, best_epoch=best_ep, epochs=epochs, lr=lr, tag=tag))
    del model; torch.cuda.empty_cache()
    return rows, preds

CONFIGS = []
for arch in ("kan", "mlp"):
    for regime in ("qwen", "llama", "mixed"):
        for seed in SEEDS:
            CONFIGS.append(("text_only", arch, regime, seed, None))
            canon = JR / "checkpoints_seed_corrected" / f"{arch}_{regime}_shared28_seed{seed}_corrected_best.pt"
            CONFIGS.append(("text_pixel", arch, regime, seed, None if RETRAIN_TEXT_PIXEL else canon))
for arch in ("mlp_pm", "linear"):
    for regime in ("qwen", "llama", "mixed"):
        for seed in SEEDS: CONFIGS.append(("text_pixel", arch, regime, seed, None))
print("configurations:", len(CONFIGS), "| to train here:", sum(1 for c in CONFIGS if c[4] is None))

t0 = time.time()
for i, (feat, arch, regime, seed, ck) in enumerate(CONFIGS, 1):
    rows, _ = run_config(feat, arch, regime, seed, ck); RES_ROWS += rows
    tl = TRAIN_LOG[-1]
    print(f"[{i:2d}/{len(CONFIGS)}] {feat:10s} {arch:7s} {regime:5s} seed{seed:<5d} params={tl['params']:6d} "
          f"{'trained' if tl['trained_here'] else 'loaded '} val={tl['best_val_acc']:.4f} | elapsed {(time.time()-t0)/60:.1f} min", flush=True)

# ---- determinism check: retrain ONE canonical Text+Pixel config and compare with the canonical checkpoint ----
if DETERMINISM_CHECK and not RETRAIN_TEXT_PIXEL:
    _, p_new = run_config("text_pixel", "kan", "qwen", 42, None, tag="_RETRAIN_CHECK")
    saved = pd.read_csv(JR / "part4_predictions_seed_corrected" / "kan_qwen_seed42_on_qwen_test.csv")
    det = dict(config="kan qwen seed42 text_pixel", label_agreement_with_canonical=float((p_new["qwen_test"]["pred_top1"].values == saved["pred_top1"].values).mean()),
               top1_retrained=float(p_new["qwen_test"]["correct"].mean()), top1_canonical=float(saved["correct"].mean()))
    json.dump(det, open(D01 / "determinism_check.json", "w"), indent=2); print(det)
    TRAIN_LOG[-1]["tag"] = "_RETRAIN_CHECK"


In [ ]:
# ===== NB1 | Collect all cell-level results (the check-run is excluded from every aggregate) =====
RES = pd.DataFrame([r for r in RES_ROWS if r.get("tag", "") == ""]).drop(columns=["tag", "macro_f1_label_set"], errors="ignore")
TRAIN = pd.DataFrame([t for t in TRAIN_LOG if t.get("tag", "") == ""])
assert len(RES.groupby(["feat", "arch", "regime", "seed"])) == 54, "expected 54 configurations (36 text/pixel-ablation + 9 MLP-PM + 9 linear)"
RES.to_csv(OUT / "all_cells_by_seed_NB1.csv", index=False); TRAIN.to_csv(OUT / "training_log_NB1.csv", index=False)
PCDF = pd.DataFrame(PC_ROWS); PCDF = PCDF[PCDF.tag == ""].drop(columns="tag")
print(RES.groupby(["feat", "arch"]).size())


In [ ]:
# ===== NB1 | Section 2: Text-only vs Text+Pixel (36 configurations, 3 seeds) =====
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
ABL = RES[RES.arch.isin(["kan", "mlp"]) & RES.feat.isin(["text_only", "text_pixel"])].copy()
MET = ["top1", "top3", "macro_f1", "weighted_f1"]
HMET = MET + ["strict_top1", "strict_top3", "ambig_top1", "ambig_top3", "macro_f1_shared26"]
agg = aggregate_seeds(ABL, ["feat", "arch", "regime", "cell"], HMET); agg.to_csv(D01 / "feature_ablation_3seed.csv", index=False)
agg[agg.cell.str.startswith("human")].to_csv(D01 / "feature_ablation_human_strata.csv", index=False)

dl = paired_delta(ABL, ["arch", "regime", "cell"], "feat", "text_pixel", "text_only", HMET)       # Text+Pixel minus Text-only, per seed
dl.to_csv(D01 / "feature_ablation_delta_per_seed.csv", index=False)
def delta_table(d, keys):
    out = []
    for k, g in d.groupby(keys, sort=False):
        rec = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        for m in ("top1", "top3", "macro_f1", "ambig_top1", "strict_top1"):
            v = g.groupby("seed")[m].mean().values * 100
            rec[f"d_{m}_pp_mean"], rec[f"d_{m}_pp_sd"], rec[f"d_{m}_seeds_positive"] = float(np.nanmean(v)), sd(v), int((v > 0).sum())
        out.append(rec)
    return pd.DataFrame(out)
delta_cfg = delta_table(dl, ["arch", "regime", "cell"]); delta_cfg.to_csv(D01 / "feature_ablation_delta_3seed.csv", index=False)
delta_pool = delta_table(dl, ["cell"]); delta_pool.to_csv(D01 / "feature_ablation_pooled_delta.csv", index=False)   # pooled over arch x regime (per-seed mean first)
delta_arch = delta_table(dl, ["arch", "cell"]); delta_arch.to_csv(D01 / "feature_ablation_delta_by_arch.csv", index=False)
delta_pool[delta_pool.cell.str.startswith("human")].to_csv(D01 / "visual_context_delta_summary.csv", index=False)

# per-class F1 (mean/SD over seeds) for every ablation prediction set
pc = PCDF[PCDF.arch.isin(["kan", "mlp"])]
pc_agg = pc.groupby(["feat", "arch", "regime", "eval", "class_id", "class_name"]).agg(f1_mean=("f1", "mean"), f1_sd=("f1", lambda x: sd(x)), support=("support", "first")).reset_index()
pc_agg.to_csv(D01 / "feature_ablation_per_class.csv", index=False)

# ---- verdict (rule-based, computed) ----
def pooled_seed_deltas(cell, metric):
    return dl[dl.cell == cell].groupby("seed")[metric].mean().values
def verdict_word(d):
    m = np.nanmean(d) * 100
    if m >= 2 and (d > 0).all(): return "improves"
    if m <= -2 and (d < 0).all(): return "degrades"
    return "shows no clear effect on"
def line(cell, metric, label):
    d = pooled_seed_deltas(cell, metric)
    return (f"- {label}: Text+Pixel minus Text-only = {np.nanmean(d)*100:+.2f} +/- {sd(d)*100:.2f} pp "
            f"(pooled over 2 heads x 3 regimes; positive in {int((d>0).sum())}/3 seeds) -> the pixel stream **{verdict_word(d)}** this measure.")
w_vis = verdict_word(pooled_seed_deltas("human_vis_yes", "top1"))
md = ["# Feature-Ablation Verdict (computed from raw predictions; 36 configurations, 3 seeds)\n",
      "Rule: an effect is called an improvement/degradation only if the pooled delta is at least 2 pp in magnitude AND has the same sign in all 3 seeds; otherwise 'no clear effect'. "
      "With n = 3 seeds the SD is a coarse estimate of training variability.\n",
      "## Pooled deltas (Text+Pixel minus Text-only)\n",
      line("qwen_test", "top1", "Qwen test Top-1"), line("llama_test", "top1", "Llama test Top-1"),
      line("human_all", "top1", "Human-written (all 372) Top-1"),
      line("human_vis_yes", "top1", "Human queries that NEED the image, Top-1"), line("human_vis_yes", "top3", "Human queries that NEED the image, Top-3"),
      line("human_vis_yes", "macro_f1", "Human queries that NEED the image, macro-F1 (present classes)"),
      line("human_vis_no", "top1", "Human queries answerable from text, Top-1"),
      line("human_new", "top1", "Human NEW-image queries, Top-1"), line("human_known", "top1", "Human KNOWN-image queries, Top-1"), "",
      "## Interpretation (neutral wording, derived from the rule above)\n"]
if w_vis == "improves":
    md.append("On human queries that need the image, adding the CLIP pixel embedding improves routing beyond the rule's threshold. This supports a limited, request-dependent role for the visual stream; it does not show that image content matters for the synthetic benchmarks unless the synthetic deltas above also improve.")
elif w_vis == "degrades":
    md.append("On human queries that need the image, adding the CLIP pixel embedding lowers accuracy beyond the rule's threshold. Report this as a negative result; the pixel stream is then kept only for architectural validity (the router receives real image evidence).")
else:
    md.append("On human queries that need the image, the pixel embedding shows no clear effect under the rule above. The visual stream should therefore be described as supplying real image evidence to the router (validity), not as improving routing accuracy.")
md.append("\nThe 'need the image' flag is an author annotation (`routing_requires_visual_context`); this is a stratified diagnostic, not a controlled test of visual reasoning.")
(D01 / "FEATURE_ABLATION_VERDICT.md").write_text("\n".join(md)); print("\n".join(md))

# ---- LaTeX tables ----
def cell_pm(feat, arch, regime, cell, m="top1"):
    r = agg[(agg.feat == feat) & (agg.arch == arch) & (agg.regime == regime) & (agg.cell == cell)].iloc[0]; return pm(r[f"{m}_mean"], r[f"{m}_sd"])
rows = []
for arch in ("kan", "mlp"):
    for regime in ("qwen", "llama", "mixed"):
        r = {"Head": arch.upper(), "Training": regime}
        for cell, lab in (("qwen_test", "Qwen test"), ("llama_test", "Llama test"), ("human_all", "Human all"), ("human_vis_yes", "Human needs-image")):
            r[f"{lab} Text"] = cell_pm("text_only", arch, regime, cell); r[f"{lab} T+P"] = cell_pm("text_pixel", arch, regime, cell)
        rows.append(r)
(D01 / "table_feature_ablation.tex").write_text(latex_table_resized(pd.DataFrame(rows), "Text-only versus Text+Pixel Top-1 (\\%), mean $\\pm$ SD over three seeds (28 classes).", "tab:feature_ablation"))
rows = []
for _, r in delta_pool.iterrows():
    rows.append({"Cell": r["cell"], "Delta Top-1 (pp)": f"{r['d_top1_pp_mean']:+.2f}$\\pm${r['d_top1_pp_sd']:.2f}", "Delta Top-3 (pp)": f"{r['d_top3_pp_mean']:+.2f}$\\pm${r['d_top3_pp_sd']:.2f}",
                 "Delta macro-F1 (pp)": f"{r['d_macro_f1_pp_mean']:+.2f}$\\pm${r['d_macro_f1_pp_sd']:.2f}", "Seeds with Delta Top-1 > 0": f"{r['d_top1_seeds_positive']}/3"})
(D01 / "table_visual_context_ablation.tex").write_text(latex_table(pd.DataFrame(rows), "Effect of the pixel stream (Text+Pixel minus Text-only), pooled over heads and regimes.", "tab:visual_context"))

# ---- figures ----
fig, axs = plt.subplots(1, 4, figsize=(17, 4)); labels = [f"{a.upper()}-{r}" for a in ("kan", "mlp") for r in ("qwen", "llama", "mixed")]
for ax, (cell, lab) in zip(axs, (("qwen_test", "Qwen test"), ("llama_test", "Llama test"), ("human_all", "Human (372)"), ("human_vis_yes", "Human: needs image"))):
    x = np.arange(6)
    for j, (feat, nm) in enumerate((("text_only", "Text-only"), ("text_pixel", "Text+Pixel"))):
        mu = [agg[(agg.feat == feat) & (agg.arch == a) & (agg.regime == r) & (agg.cell == cell)].top1_mean.iloc[0] for a in ("kan", "mlp") for r in ("qwen", "llama", "mixed")]
        sdv = [agg[(agg.feat == feat) & (agg.arch == a) & (agg.regime == r) & (agg.cell == cell)].top1_sd.iloc[0] for a in ("kan", "mlp") for r in ("qwen", "llama", "mixed")]
        ax.bar(x + (j - 0.5) * 0.38, mu, 0.38, yerr=sdv, capsize=2, label=nm)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8); ax.set_ylim(0, 1); ax.set_title(lab, fontsize=10); ax.set_ylabel("Top-1")
axs[0].legend(fontsize=8); fig.tight_layout(); fig.savefig(D01 / "fig_feature_ablation_final.png", dpi=150); plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 4)); cells = ["qwen_test", "llama_test", "human_all", "human_vis_yes", "human_vis_no", "human_known", "human_new"]
for a, c in (("kan", "tab:blue"), ("mlp", "tab:orange")):
    mu = [delta_arch[(delta_arch.arch == a) & (delta_arch.cell == k)].d_top1_pp_mean.iloc[0] for k in cells]; e = [delta_arch[(delta_arch.arch == a) & (delta_arch.cell == k)].d_top1_pp_sd.iloc[0] for k in cells]
    ax.errorbar(np.arange(len(cells)) + (0.08 if a == "mlp" else -0.08), mu, yerr=e, fmt="o", color=c, capsize=3, label=a.upper())
ax.axhline(0, color="gray", lw=0.8); ax.set_xticks(range(len(cells))); ax.set_xticklabels(cells, rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Text+Pixel minus Text-only, Top-1 (pp)"); ax.legend(); fig.tight_layout(); fig.savefig(D01 / "fig_visual_context_final.png", dpi=150); plt.close(fig)
print("ablation outputs written to", D01)


In [ ]:
# ===== NB1 | Section 3: parameter-matched MLP vs KAN (and vs the original MLP) =====
RP = RES[RES.feat == "text_pixel"].copy()
aggregate_seeds(RP[RP.arch == "mlp_pm"], ["arch", "regime", "cell"], HMET).to_csv(D02 / "mlp_pm_3seed_results.csv", index=False)
cmp_pm = pd.concat([pooled_compare(RP, "arch", "kan", "mlp_pm").assign(scope="all28"), pooled_compare(RP, "arch", "kan", "mlp").assign(scope="all28"),
                    pooled_compare(RP, "arch", "mlp", "mlp_pm").assign(scope="all28"),
                    pooled_compare(RP, "arch", "kan", "mlp_pm", ("top1", "macro_f1"), GROUPS_STABLE).assign(scope="stable21"),
                    pooled_compare(RP, "arch", "kan", "mlp", ("top1", "macro_f1"), GROUPS_STABLE).assign(scope="stable21")], ignore_index=True)
cmp_pm.to_csv(D02 / "mlp_pm_comparison_to_kan.csv", index=False)
cells_cmp = paired_delta(RP[RP.arch.isin(["kan", "mlp_pm"])], ["regime", "cell"], "arch", "kan", "mlp_pm", HMET)
cells_cmp.groupby(["regime", "cell"])[HMET].agg(["mean", "std"]).to_csv(D02 / "mlp_pm_vs_kan_cells_delta.csv")
arch_spec = dict(kan=dict(projection=48, kan_widths=[48, 48, 28], grid=4, params=PARAMS["kan"]),
                 mlp_original=dict(projection=96, hidden=[64, 32], dropout=0.3, params=PARAMS["mlp"]),
                 mlp_param_matched=dict(projection=MLP_PM_PROJ, hidden=[64, 32], dropout=0.3, params=PARAMS["mlp_pm"],
                                        relative_to_kan_pct=100 * (PARAMS["mlp_pm"] - PARAMS["kan"]) / PARAMS["kan"]),
                 note="Same input, optimiser, scheduler, label smoothing, batch size, seeds and checkpoint rule as the original MLP; only the projection width changes.")
json.dump(arch_spec, open(D02 / "mlp_pm_architecture.json", "w"), indent=2)
(D02 / "parameter_count_check.txt").write_text("\n".join(f"{k}: {v:,}" for k, v in PARAMS.items()) + f"\nMLP-PM vs KAN: {arch_spec['mlp_param_matched']['relative_to_kan_pct']:+.2f}%\n")

def g(cmp, a, b, group, metric, scope="all28"):
    r = cmp[(cmp.a == a) & (cmp.b == b) & (cmp.group == group) & (cmp.metric == metric) & (cmp.scope == scope)].iloc[0]; return r
def fmtrow(r): return f"{r.diff_mean*100:+.2f} +/- {r.diff_sd*100:.2f} pp (KAN ahead in {r.seeds_a_ahead}/3 seeds)"
md = ["# Parameter-Matched MLP Verdict (computed)\n", f"KAN = {PARAMS['kan']:,} trainable parameters; MLP-PM = {PARAMS['mlp_pm']:,} ({arch_spec['mlp_param_matched']['relative_to_kan_pct']:+.2f}%); original MLP = {PARAMS['mlp']:,}.\n",
      "## KAN minus MLP-PM (pooled evaluation groups; seed-paired mean +/- SD)\n"]
for grp in GROUPS:
    md.append(f"- **{grp}**: Top-1 {fmtrow(g(cmp_pm,'kan','mlp_pm',grp,'top1'))}; Top-3 {fmtrow(g(cmp_pm,'kan','mlp_pm',grp,'top3'))}; macro-F1 {fmtrow(g(cmp_pm,'kan','mlp_pm',grp,'macro_f1'))}")
md.append("\n## Same comparison on the 21 stable-support classes\n")
for grp in GROUPS_STABLE:
    md.append(f"- **{grp}**: Top-1 {fmtrow(g(cmp_pm,'kan','mlp_pm',grp,'top1','stable21'))}; macro-F1 {fmtrow(g(cmp_pm,'kan','mlp_pm',grp,'macro_f1','stable21'))}")
top1_adv = sum(1 for grp in GROUPS if (lambda r: r.diff_mean > 0.01 and r.seeds_a_ahead == 3)(g(cmp_pm, "kan", "mlp_pm", grp, "top1")))
f1_adv = sum(1 for grp in GROUPS if (lambda r: r.diff_mean > 0.01 and r.seeds_a_ahead == 3)(g(cmp_pm, "kan", "mlp_pm", grp, "macro_f1")))
f1_adv_st = sum(1 for grp in GROUPS_STABLE if (lambda r: r.diff_mean > 0.01 and r.seeds_a_ahead == 3)(g(cmp_pm, "kan", "mlp_pm", grp, "macro_f1", "stable21")))
md += ["\n## Answers (rule: an advantage needs >= 1 pp pooled difference and KAN ahead in 3/3 seeds)\n",
       f"1. KAN keeps a Top-1 advantage over the capacity-matched MLP in {top1_adv}/5 evaluation groups; a macro-F1 advantage in {f1_adv}/5 groups on all 28 classes and in {f1_adv_st}/3 groups on the stable-support set.",
       f"2. Size advantage: {'none - the two heads are within ' + format(abs(arch_spec['mlp_param_matched']['relative_to_kan_pct']), '.2f') + '% of each other' if abs(arch_spec['mlp_param_matched']['relative_to_kan_pct']) < 2 else 'present'}. The earlier '35.6% fewer parameters' statement compared against the larger original MLP and must be reworded.",
       f"3. Remaining distinction of KAN-Route: {'structural (inspectable edge functions) rather than an accuracy or size advantage' if top1_adv == 0 and f1_adv_st == 0 else 'partly empirical; see the groups listed above'}.",
       "\nThe heads still differ in learning rate and epochs (KAN 5e-4/80, MLPs 1e-3/60); this is stated as a limitation, not equalised."]
(D02 / "PARAMETER_MATCHED_VERDICT.md").write_text("\n".join(md)); print("\n".join(md))

rows = []
for grp in GROUPS:
    r = {"Group": grp}
    for mod, lab in (("kan", "KAN"), ("mlp_pm", "MLP-PM"), ("mlp", "MLP")):
        v = pooled_group(RP, "arch", mod, grp, "top1"); f = pooled_group(RP, "arch", mod, grp, "macro_f1")
        r[f"{lab} Top-1"] = pm(v.mean(), sd(v)); r[f"{lab} macro-F1"] = pm(f.mean(), sd(f))
    r["KAN-MLP-PM Top-1 (pp)"] = f"{g(cmp_pm,'kan','mlp_pm',grp,'top1').diff_mean*100:+.2f}"; r["KAN-MLP-PM macro-F1 (pp)"] = f"{g(cmp_pm,'kan','mlp_pm',grp,'macro_f1').diff_mean*100:+.2f}"
    rows.append(r)
(D02 / "table_kan_vs_mlp_pm.tex").write_text(latex_table_resized(pd.DataFrame(rows), f"KAN-Route versus a parameter-matched MLP ({PARAMS['mlp_pm']:,} vs {PARAMS['kan']:,} parameters), pooled groups, mean $\\pm$ SD over three seeds.", "tab:kan_vs_mlp_pm"))


In [ ]:
# ===== NB1 | Section 4: linear baseline and non-linear gain =====
aggregate_seeds(RP[RP.arch == "linear"], ["arch", "regime", "cell"], HMET).to_csv(D03 / "linear_3seed_results.csv", index=False)
gain = pd.concat([pooled_compare(RP, "arch", m, "linear").assign(scope="all28") for m in ("kan", "mlp", "mlp_pm")] +
                 [pooled_compare(RP, "arch", m, "linear", ("top1", "macro_f1"), GROUPS_STABLE).assign(scope="stable21") for m in ("kan", "mlp", "mlp_pm")], ignore_index=True)
gain.to_csv(D03 / "nonlinear_gain.csv", index=False)
(D03 / "linear_baseline_architecture.txt").write_text(f"Linear head: nn.Linear(896, 28), {PARAMS['linear']:,} trainable parameters; AdamW lr 1e-3, 60 epochs, one-cycle, label smoothing 0.05, batch 256, best-validation checkpoint (no tuning).\n")
md = ["# Linear Baseline Verdict (computed)\n", f"Linear head: {PARAMS['linear']:,} parameters (no hidden layer). Gain = head minus linear, pooled groups, seed-paired mean +/- SD.\n"]
for m, lab in (("kan", "KAN"), ("mlp", "MLP"), ("mlp_pm", "MLP-PM")):
    md.append(f"\n### {lab} minus Linear")
    for grp in GROUPS:
        a, b = g(gain, m, "linear", grp, "top1"), g(gain, m, "linear", grp, "macro_f1")
        md.append(f"- {grp}: Top-1 {a.diff_mean*100:+.2f} +/- {a.diff_sd*100:.2f} pp ({a.seeds_a_ahead}/3 seeds ahead); macro-F1 {b.diff_mean*100:+.2f} +/- {b.diff_sd*100:.2f} pp")
need = {m: sum(1 for grp in GROUPS if (lambda r: r.diff_mean >= 0.02 and r.seeds_a_ahead == 3)(g(gain, m, "linear", grp, "top1"))) for m in ("kan", "mlp", "mlp_pm")}
md += ["\n## Answer (rule: gain needs >= 2 pp pooled Top-1 and 3/3 seeds)\n",
       f"Groups (of 5) where a non-linear head beats the linear head on Top-1: KAN {need['kan']}, MLP {need['mlp']}, MLP-PM {need['mlp_pm']}.",
       "If the linear head is within about 2 pp of the non-linear heads, the frozen MiniLM+CLIP features are close to linearly separable for this taxonomy, and head capacity is not what limits routing; if the gap is large, non-linearity matters. State whichever the numbers above show."]
(D03 / "LINEAR_BASELINE_VERDICT.md").write_text("\n".join(md)); print("\n".join(md))
rows = []
for grp in GROUPS:
    r = {"Group": grp}
    for mod, lab in (("linear", "Linear"), ("mlp_pm", "MLP-PM"), ("mlp", "MLP"), ("kan", "KAN")):
        v = pooled_group(RP, "arch", mod, grp, "top1"); r[f"{lab} Top-1"] = pm(v.mean(), sd(v))
    rows.append(r)
(D03 / "table_linear_baseline.tex").write_text(latex_table_resized(pd.DataFrame(rows), "Linear baseline versus the non-linear heads, Top-1 (\\%), pooled groups, mean $\\pm$ SD over three seeds.", "tab:linear_baseline"))


In [ ]:
# ===== NB1 | Save per-class + package =====
PCDF.groupby(["feat", "arch", "regime", "eval", "class_id", "class_name"]).agg(f1_mean=("f1", "mean"), f1_sd=("f1", lambda x: sd(x)), support=("support", "first")).reset_index().to_csv(OUT / "per_class_all_models_NB1.csv", index=False)
(OUT / "README_NB1.md").write_text(f"NB1 outputs: 00 frozen manifest, 01 feature ablation, 02 parameter-matched MLP, 03 linear baseline.\nGPU: {GPU_NAME}. RETRAIN_TEXT_PIXEL={RETRAIN_TEXT_PIXEL}. MLP-PM projection={MLP_PM_PROJ}.\n")
shutil.make_archive("/kaggle/working/NB1_OUTPUT", "zip", OUT)
print("DONE. Download /kaggle/working/NB1_OUTPUT.zip, or use 'Save Version' and attach this notebook's output to NB3.")
